# 02 — Agentul conversațional (Telegram)

**Echipa:** Tofan Bogdan, Manea Alina-Alexandra, Burcă Alina  
**Materia:** NLP

Acest notebook reproduce flow-ul `Conversational agent` din N8N.

## Ce face

1. Primește un mesaj **text sau voce** pe bot-ul Telegram.
2. Dacă e voce: descarcă fișierul și îl transcrie cu **Whisper** (OpenAI).
3. Trimite textul către agentul **Grok** care:
   - are **memorie de conversație** în MongoDB (`Chat_history`)
   - interoghează **Pinecone** (`fin-news-documents`) pentru context din știrile istorice
4. Răspunde în **aceeași modalitate** (text dacă input-ul a fost text, audio TTS dacă a fost voce).

Notebook-ul are două părți:
- **Test sandbox** — invocă agentul direct, fără Telegram, pentru iterație rapidă.
- **Live polling** — pornește bot-ul real (rulează până oprești kernel-ul).

## 1. Setup

In [ ]:
# !git clone https://github.com/BogdanT54/financial-news-agent.git
# %cd financial-news-agent
# !pip install -q -r requirements.txt

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.insert(0, os.path.abspath(".."))

from src.config import get_settings
settings = get_settings()
print("Pinecone index:", settings.PINECONE_INDEX)
print("Main model    :", settings.MODEL_MAIN_AGENT)
print("DRY_RUN       :", settings.DRY_RUN)

## 2. Construiește agentul conversațional

Agent = LLM (Grok via OpenRouter) + tool Pinecone (retrieve top 50 articole istorice).

In [ ]:
from src.vectorstore import as_retriever_tool
from src.agents import build_conversational_agent, run_conversational_agent

retriever_tool = as_retriever_tool(top_k=50)
executor = build_conversational_agent(retriever_tool)
print("Agent gata.")

## 3. Test rapid cu input text

Întreabă orice despre piețe, crypto, macro etc. — agentul va interoga Pinecone și va răspunde.

In [ ]:
raspuns = run_conversational_agent(
    executor,
    user_message="Care este sentimentul recent pe Bitcoin? Dă-mi 2-3 referințe din baza de date.",
)
print(raspuns)

## 4. Test cu input audio (upload .ogg / .mp3)

Încarcă un fișier audio și agentul îl va transcrie cu Whisper, va răspunde și va genera audio TTS.

In [ ]:
from src.tts import transcribe_audio, generate_audio_opus
from IPython.display import Audio, display

# Înlocuiește cu path-ul către fișierul tău audio
AUDIO_PATH = "sample.ogg"

if os.path.exists(AUDIO_PATH):
    with open(AUDIO_PATH, "rb") as f:
        audio_in = f.read()
    transcript = transcribe_audio(audio_in, filename=os.path.basename(AUDIO_PATH))
    print("Transcript :", transcript)

    raspuns = run_conversational_agent(executor, user_message=transcript)
    print("\nRăspuns text:\n", raspuns)

    audio_out = generate_audio_opus(raspuns)
    display(Audio(data=audio_out, autoplay=False))
else:
    print(f"Nu există {AUDIO_PATH}. Încarcă un fișier audio și re-rulează celula.")

## 5. Live bot polling (opțional)

Celula de mai jos pornește bot-ul Telegram în polling. Va rula până oprești kernel-ul. 

**Atenție:** pe Colab este nevoie de `nest_asyncio` pentru că loop-ul async se ciocnește cu cel al notebook-ului.

In [ ]:
# !pip install -q nest_asyncio

# import nest_asyncio; nest_asyncio.apply()
# from src.telegram_io import run_bot_polling, download_voice
# from telegram import Update
# from telegram.ext import ContextTypes
# from src.tts import transcribe_audio, generate_audio_opus
#
# async def handler(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
#     msg = update.message
#     if msg is None:
#         return
#     await context.bot.send_message(chat_id=msg.chat.id, text="Processing...",
#                                    reply_to_message_id=msg.message_id)
#     if msg.voice:
#         audio_bytes = await download_voice(msg.voice.file_id)
#         text_in = transcribe_audio(audio_bytes, filename="voice.ogg")
#         reply = run_conversational_agent(executor, user_message=text_in)
#         audio_out = generate_audio_opus(reply)
#         await context.bot.send_audio(chat_id=msg.chat.id, audio=audio_out,
#                                      filename="reply.opus",
#                                      reply_to_message_id=msg.message_id)
#     else:
#         reply = run_conversational_agent(executor, user_message=msg.text or "")
#         await context.bot.send_message(chat_id=msg.chat.id, text=reply,
#                                        reply_to_message_id=msg.message_id)
#
# run_bot_polling(handler)